# Production-Style End-to-End Demo

This notebook stitches together core capabilities for a production-like workflow:

- Namespaced agents with guarded tools and memories.
- `with_steps` for batch/parallel/conditional logic inside an agent.
- `GraphRunner` for DAG orchestration across namespaces.
- CommunicationHub for direct, broadcast, and namespace messaging.
- RunStore for persistence; metrics/error/latency recorded in context.
- Safe tools: sanitized code exec, safe file read, MCP registry (stub), RL-aware tools.

Set `OPENAI_API_KEY` at runtime to hit OpenAI; otherwise a stub LLM is used.

In [1]:
import os
from getpass import getpass
from pathlib import Path

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY (press Enter to skip): ")

from agentic_codex import (
    AgentBuilder,
    Context,
    GraphRunner,
    GraphNodeSpec,
    StepSpec,
    FunctionAdapter,
    EnvOpenAIAdapter,
    CommunicationHub,
    CommEnvelope,
    RunStore,
    MCPServerConfig,
    MCPToolDescriptor,
    MCPToolRegistry,
    ToolCapability,
)
from agentic_codex.core.schemas import AgentStep, Message
from agentic_codex.core.tools import (
    SafeFileReaderToolAdapter,
    SanitizedCodeExecutionToolAdapter,
    ReinforcementToolAdapter,
    MathToolAdapter,
)

def stub_llm(prompt: str) -> str:
    return f"[stub llm] {prompt[:120]}"

try:
    llm = EnvOpenAIAdapter(model="gpt-4o-mini")
except Exception:
    llm = FunctionAdapter(stub_llm)

# --- Communication and namespaces ---
hub = CommunicationHub()
hub.register_participant("planner")
hub.register_participant("builder")
hub.register_participant("qa")
hub.create_namespace("delivery", ["planner", "builder"])
hub.create_namespace("quality", ["qa"])
hub.link_namespaces("delivery", "quality")

# --- MCP registry stub ---
mcp_server = MCPServerConfig(name="local", base_url="https://mcp.local")
mcp_registry = MCPToolRegistry(mcp_server)
mcp_registry.register(MCPToolDescriptor(name="docs.search", description="Search docs", path="/search"))

# --- Safe tools ---
safe_reader = SafeFileReaderToolAdapter(root=Path("."))
safe_code = SanitizedCodeExecutionToolAdapter()
rl_math = ReinforcementToolAdapter(MathToolAdapter(), reward_fn=lambda res: -abs(res["result"]))

# --- Agent builders ---
def make_agent(name: str, role: str):
    def step(ctx: Context) -> AgentStep:
        msg = f"{name}:{role}:{ctx.goal}:{ctx.scratch.get('batch_item','')}"
        hub.send(CommEnvelope(sender=name, targets=["qa"], mode="namespace", content=msg))
        return AgentStep(out_messages=[Message(role=name, content=msg)], state_updates={name: True})

    return (
        AgentBuilder(name=name, role=role)
        .with_llm(llm)
        .with_capability(ToolCapability({
            "fs.safe_read": safe_reader,
            "code.sanitized_exec": safe_code,
            "math.rl": rl_math,
        }))
        .with_capability(mcp_registry.as_capability())
        .with_step(step)
        .build()
    )

planner = make_agent("planner", "planner")
builder_agent = make_agent("builder", "executor")
qa_agent = make_agent("qa", "quality")

# --- with_steps inside builder_agent to batch tasks ---
builder_steps = [
    StepSpec(name="prepare", fn=builder_agent.run),
    StepSpec(name="tasks", fn=builder_agent.run, batch_key="tasks", parallel=True, isolate_state=True),
]
builder_with_steps = AgentBuilder(name="builder_with_steps", role="executor").with_llm(llm).with_steps(builder_steps).build()

# --- Graph orchestration ---
graph = {
    "plan": GraphNodeSpec(agent=planner),
    "build": GraphNodeSpec(agent=builder_with_steps, after=["plan"]),
    "qa": GraphNodeSpec(agent=qa_agent, after=["build"], condition=lambda ctx: True),
}

ctx = Context(goal="Ship prod feature", scratch={"tasks": ["design", "impl", "test"]})
runner = GraphRunner(graph, allow_parallel=True, isolate_parallel_state=True)
run_result = runner.run(goal=ctx.goal, inputs=ctx.scratch)

# Persist run
store = RunStore("runs/")
stored_path = store.save(run_result)

history = hub.history()
ephemeral = hub.ephemeral()
metrics = ctx.scratch.get("_metrics", {})
len(run_result.messages), len(history), len(ephemeral), stored_path, list(metrics.keys())

(6, 0, 0, PosixPath('runs/d8c1fafe5f81408683342c50347de7df.json'), [])